In [ ]:
import numpy as np
import tensorflow as tf
import random
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SEQ_LEN = 10
TARGET_COL = "label"

CLASSES = ["air", "cinnamon", "rosemary"]
NUM_CLASSES = len(CLASSES)

FEATURE_COLS = [
    "humidity_z",
    "log_ratio",
]

In [2]:
# %%
def load_dataset(base_dir: str) -> pd.DataFrame:
    dfs = []
    session_id = 0

    for class_folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, class_folder)

        if not os.path.isdir(folder_path):
            continue

        for file in os.listdir(folder_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            df["session_id"] = session_id

            df["id"] = df.apply(
                lambda x: f"{session_id}_{int(x['sensor_index'])}_{int(x['fingerprint_index'])}",
                axis=1,
            )

            dfs.append(df)
            session_id += 1

    if not dfs:
        raise ValueError("No CSV files found.")

    return pd.concat(dfs, ignore_index=True)


data = load_dataset("../data/new-dataset")

# Keep only stable rows
data = data[data["label"].str.endswith("Stable")].copy()

# Convert labels
data["label"] = data["label"].replace({"baselineStable": "air"})
data["label"] = data["label"].str.replace("Stable", "", regex=False)

print("Sessions:", data["session_id"].nunique())

Sessions: 9


In [3]:
def compute_baseline(df: pd.DataFrame) -> pd.DataFrame:
    baseline = (
        df[df["label"] == "air"]
        .groupby(["session_id", "sensor_index", "position"])["gas_resistance"]
        .mean()
        .rename("R_base")
        .reset_index()
    )

    df = df.merge(
        baseline,
        on=["session_id", "sensor_index", "position"],
        how="left"
    )

    if df["R_base"].isna().any():
        missing = df[df["R_base"].isna()]["session_id"].unique()
        raise ValueError(f"Missing air baseline in sessions: {missing}")

    return df

data = compute_baseline(data)
data.head()

,sensor_index,fingerprint_index,position,plate_temperature,heater_duration,temperature,pressure,humidity,gas_resistance,label,session_id,id,R_base
0,3,31,1,150,180,33.18,1033.76,37.09,7156148,air,0,0_3_31,7.090028e+06
1,1,30,7,380,140,34.07,1029.70,27.09,88306,air,0,0_1_30,8.718150e+04
2,8,29,7,380,140,32.91,1010.61,25.76,69245,air,0,0_8_29,6.875067e+04
3,7,32,3,250,150,33.37,1063.82,32.17,253905,air,0,0_7_32,2.499395e+05
4,5,29,7,380,140,32.70,1112.03,34.55,91527,air,0,0_5_29,9.114756e+04


In [4]:
from sklearn.preprocessing import LabelEncoder

sessions = data["session_id"].unique()

train_sessions, temp_sessions = train_test_split(
    sessions,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

val_sessions, test_sessions = train_test_split(
    temp_sessions,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

train_data = data[data["session_id"].isin(train_sessions)].copy()
val_data   = data[data["session_id"].isin(val_sessions)].copy()
test_data  = data[data["session_id"].isin(test_sessions)].copy()

label_encoder = LabelEncoder()

train_data["label_enc"] = label_encoder.fit_transform(train_data[TARGET_COL])
val_data["label_enc"] = label_encoder.transform(val_data[TARGET_COL])
test_data["label_enc"] = label_encoder.transform(test_data[TARGET_COL])

print("Train sessions:", len(train_sessions))
print("Val sessions  :", len(val_sessions))
print("Test sessions :", len(test_sessions))

Train sessions: 6
Val sessions  : 1
Test sessions : 2


In [5]:
train_data["log_ratio"] = np.log(
    train_data["gas_resistance"] / train_data["R_base"]
)

val_data["log_ratio"] = np.log(
    val_data["gas_resistance"] / val_data["R_base"]
)

test_data["log_ratio"] = np.log(
    test_data["gas_resistance"] / test_data["R_base"]
)

In [6]:
env_cols = ["humidity"]

for col in env_cols:
    mean = train_data.groupby("sensor_index")[col].mean()
    std  = train_data.groupby("sensor_index")[col].std()

    train_data[f"{col}_z"] = (
        train_data[col] - train_data["sensor_index"].map(mean)
    ) / train_data["sensor_index"].map(std)

    val_data[f"{col}_z"] = (
        val_data[col] - val_data["sensor_index"].map(mean)
    ) / val_data["sensor_index"].map(std)

    test_data[f"{col}_z"] = (
        test_data[col] - test_data["sensor_index"].map(mean)
    ) / test_data["sensor_index"].map(std)

In [7]:
def keep_complete(df):
    valid_ids = (
        df.groupby("id")
          .size()
          .loc[lambda x: x == SEQ_LEN]
          .index
    )
    return df[df["id"].isin(valid_ids)]

train_data = keep_complete(train_data)
val_data = keep_complete(val_data)
test_data  = keep_complete(test_data)

print("Train fingerprints:", train_data["id"].nunique())
print("Val fingerprints:", val_data["id"].nunique())
print("Test fingerprints :", test_data["id"].nunique())

Train fingerprints: 1173
Val fingerprints: 113
Test fingerprints : 450


In [8]:
train_data

,sensor_index,fingerprint_index,position,plate_temperature,heater_duration,temperature,pressure,humidity,gas_resistance,label,session_id,id,R_base,label_enc,log_ratio,humidity_z
23,1,31,0,100,200,34.12,1029.72,27.04,20183554,air,0,0_1_31,1.987855e+07,0,0.015227,-0.298293
29,5,30,0,100,200,32.73,1112.03,34.59,24417288,air,0,0_5_30,2.412449e+07,0,0.012064,-0.126842
31,1,31,1,150,180,34.11,1029.71,27.03,3765571,air,0,0_1_31,3.745569e+06,0,0.005326,-0.300921
35,4,32,0,100,200,32.69,1103.66,34.75,31155692,air,0,0_4_32,3.078030e+07,0,0.012122,-0.426547
36,2,32,0,100,200,33.63,1106.62,36.10,30439386,air,0,0_2_32,3.075996e+07,0,-0.010476,-0.365445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23888,2,82,9,450,140,33.17,1104.67,38.31,74832,rosemary,8,8_2_82,1.290499e+05,2,-0.544953,0.177477
23890,4,82,9,450,140,32.76,1101.75,36.36,74832,rosemary,8,8_4_82,1.321526e+05,2,-0.568712,-0.147306
23892,3,81,8,415,140,33.33,1031.89,38.70,70914,rosemary,8,8_3_81,1.402898e+05,2,-0.682242,-0.188539
23894,7,84,9,450,140,35.95,1061.99,30.79,62730,rosemary,8,8_7_84,1.114328e+05,2,-0.574582,-0.183082


In [9]:
import numpy as np

def create_sequences_grouped(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    seq_len: int
):
    X, y = [], []
    grouped = df.groupby("id")
    num_skipped = 0
    for _id, group in grouped:
        group = group.sort_values("position")
        data = group[feature_cols].values

        if len(data) == seq_len:
            X.append(data)
            y.append(int(group[target_col].iloc[0]))
        else:
            num_skipped += 1
    print("Number of sequences skipped: ", num_skipped)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


X_train, y_train = create_sequences_grouped(train_data, FEATURE_COLS, 'label_enc', SEQ_LEN)
X_val, y_val = create_sequences_grouped(val_data, FEATURE_COLS, 'label_enc', SEQ_LEN)
X_test, y_test = create_sequences_grouped(test_data, FEATURE_COLS, 'label_enc', SEQ_LEN)

Number of sequences skipped:  0
Number of sequences skipped:  0
Number of sequences skipped:  0


In [10]:
from keras.models import Sequential
from keras.layers import Conv1D, Dense, Dropout, Input, GlobalAveragePooling1D

model = Sequential([
    Input(shape=(SEQ_LEN, len(FEATURE_COLS))),

    Conv1D(16, kernel_size=3, activation='relu'),
    Conv1D(16, kernel_size=3, activation='relu'),

    GlobalAveragePooling1D(),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 8, 16)          │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 6, 16)          │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,491 (5.82 KB)

 Trainable params: 1,491 (5.82 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True
)

Epoch 1/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3112 - loss: 1.1110 - val_accuracy: 0.8761 - val_loss: 1.0003
Epoch 2/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5524 - loss: 1.0358 - val_accuracy: 0.8850 - val_loss: 0.7033
Epoch 3/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5857 - loss: 0.9551 - val_accuracy: 0.8761 - val_loss: 0.4812
Epoch 4/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5857 - loss: 0.8795 - val_accuracy: 0.8496 - val_loss: 0.3764
Epoch 5/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5916 - loss: 0.8250 - val_accuracy: 0.8673 - val_loss: 0.3549
Epoch 6/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6155 - loss: 0.7959 - val_accuracy: 0.7876 - val_loss: 0.3654
Epoch 7/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6445 - loss: 0.7836 - val_accuracy: 0.7876 - val_loss: 0.3777
Epoch 8/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6513 - loss: 0.7759 - val_accuracy: 0.7788 - val_loss:

In [12]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

overall_acc = accuracy_score(y_test, y_pred)
print("Overall Test Accuracy:", overall_acc)

# Confusion matrix for all classes (force full label range)
labels_full = np.arange(NUM_CLASSES)
cm = confusion_matrix(y_test, y_pred, labels=labels_full)

# per-class accuracy (handle zero samples)
denom = cm.sum(axis=1)
per_class_acc = np.divide(cm.diagonal(), denom,
                          out=np.zeros_like(cm.diagonal(), dtype=float),
                          where=denom!=0)

class_names = label_encoder.inverse_transform(labels_full)

for name, acc, n in zip(class_names, per_class_acc, denom):
    print(f"{name:20s} : {acc:.3f}  (n_test={int(n)})")

print("\nClassification Report (only shows classes present in y_test):")
print(classification_report(y_test, y_pred, labels=np.unique(y_test),
                            target_names=label_encoder.inverse_transform(np.unique(y_test))))


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Overall Test Accuracy: 0.5511111111111111
air                  : 0.976  (n_test=168)
cinnamon             : 1.000  (n_test=84)
rosemary             : 0.000  (n_test=198)

Classification Report (only shows classes present in y_test):
              precision    recall  f1-score   support

         air       0.45      0.98      0.62       168
    cinnamon       0.95      1.00      0.98        84
    rosemary       0.00      0.00      0.00       198

    accuracy                           0.55       450
   macro avg       0.47      0.66      0.53       450
weighted avg       0.35      0.55      0.41       450



d:\Archived Projects\edinburghai-expo\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Archived Projects\edinburghai-expo\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Archived Projects\edinburghai-expo\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [13]:
disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    display_labels=label_encoder.inverse_transform(np.arange(NUM_CLASSES)),
    normalize='true',   # key line
    xticks_rotation=45
)

plt.title("Normalized Confusion Matrix")
plt.show()

NameError: name 'ConfusionMatrixDisplay' is not defined

In [ ]:
def evaluate_csv(file_path: str):
    import pandas as pd
    import numpy as np

    print(f"\nEvaluating: {file_path}")

    # Load
    df = pd.read_csv(file_path)

    # --- Match your preprocessing ---

    # Keep only stable rows
    df = df[df["label"].str.endswith("Stable")].copy()

    # Normalize labels
    df["label"] = df["label"].replace({"baselineStable": "air"})
    df["label"] = df["label"].str.replace("Stable", "", regex=False)

    # Fake session_id (single session)
    df["session_id"] = 0

    # Create ID (same as training)
    df["id"] = df.apply(
        lambda x: f"0_{int(x['sensor_index'])}_{int(x['fingerprint_index'])}",
        axis=1,
    )

    # --- Compute baseline (same logic) ---
    baseline = (
        df[df["label"] == "air"]
        .groupby(["session_id", "sensor_index", "position"])["gas_resistance"]
        .mean()
        .rename("R_base")
        .reset_index()
    )

    df = df.merge(
        baseline,
        on=["session_id", "sensor_index", "position"],
        how="left"
    )

    if df["R_base"].isna().any():
        raise ValueError("Missing air baseline in this CSV")

    # --- Feature engineering ---
    df["log_ratio"] = np.log(df["gas_resistance"] / df["R_base"])

    # Use TRAIN stats (must exist in memory!)
    for col in ["humidity"]:
        df[f"{col}_z"] = (
            df[col] - df["sensor_index"].map(train_data.groupby("sensor_index")[col].mean())
        ) / df["sensor_index"].map(train_data.groupby("sensor_index")[col].std())

    # Encode labels
    df["label_enc"] = label_encoder.transform(df["label"])

    # --- Keep valid sequences ---
    def keep_complete(df):
        valid_ids = (
            df.groupby("id")
              .size()
              .loc[lambda x: x == SEQ_LEN]
              .index
        )
        return df[df["id"].isin(valid_ids)]

    df = keep_complete(df)

    # --- Create sequences ---
    def create_sequences(df):
        X, y = [], []
        for _id, group in df.groupby("id"):
            group = group.sort_values("position")
            data = group[FEATURE_COLS].values

            if len(data) == SEQ_LEN:
                X.append(data)
                y.append(int(group["label_enc"].iloc[0]))

        return np.array(X, dtype=np.float32), np.array(y)

    X, y = create_sequences(df)

    if len(X) == 0:
        print("No valid sequences found.")
        return

    # --- Predict ---
    y_pred = np.argmax(model.predict(X), axis=1)

    acc = (y_pred == y).mean()
    print(f"Accuracy: {acc:.4f}")

    return acc

evaluate_csv("../data/test/rosemary.csv")

In [ ]:
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")